# Avaluació offline sobre `shop.history_sessions`

Còpia adaptada de `starter/offline_eval.ipynb`. Abans de tocar `src/recommender.py`,
qualsevol idea nova es valida aquí.

**Abans de córrer aquest notebook**, genereu els models amb el mateix tall de dates
que useu com a `CUT` més avall:

```bash
python ../training/train_model.py --until 2026-11-10
```

(ajusteu la data al que imprimeixi la cel·la següent; és el tall real que separa
entrenament de test aquí sota).

In [1]:
import sys, pickle
from collections import Counter
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import numpy as np
import pandas as pd
from src.rawtree_client import sql, sql_df

MODELS_DIR = Path("..") / "models"

In [2]:
last_day = sql_df("SELECT toDate(max(ts::DateTime64(3))) AS d FROM shop.history_sessions").d[0]
CUT = str((pd.Timestamp(last_day) - pd.Timedelta(days=7)).date())
print("train <", CUT, "<= test   ->   correu (si no ho heu fet):")
print(f"  python ../training/train_model.py --until {CUT}")

test = sql_df(f"""
    SELECT request_id, ts, user_id, context, recent_views, shown, propensity, clicked, purchased
    FROM shop.history_sessions WHERE ts >= '{CUT}' ORDER BY rand() LIMIT 100000""")
test["clicked"] = test.clicked.replace("", None)
print(len(test), "impressions de test, CTR observat", round(test.clicked.notna().mean(), 4))

train < 2026-11-07 <= test   ->   correu (si no ho heu fet):
  python ../training/train_model.py --until 2026-11-07
100000 impressions de test, CTR observat 0.0373


## Carregar els models entrenats amb `--until CUT`

Si un fitxer no existeix, el diccionari/llista queda buit i la policy corresponent
cau automàticament cap al baseline per a aquella part — no peta, però tampoc millora res.

In [3]:
def load(name, default):
    path = MODELS_DIR / f"{name}_{CUT}.pkl"
    if not path.exists():
        print(f"AVÍS: {path.name} no existeix -> correu training/train_model.py --until {CUT}")
        return default
    with open(path, "rb") as f:
        return pickle.load(f)

best = load("best_sellers", [])
covisit = dict(load("co_purchases", {}))
user_profiles = dict(load("user_profiles", {}))
print(len(best), "best sellers,", len(covisit), "productes amb co-compres,", len(user_profiles), "perfils d'usuari")

4787 best sellers, 5000 productes amb co-compres, 57513 perfils d'usuari


## Tendència regional, aproximada amb dades anteriors al tall

No tenim una taula "última hora" dins l'històric, així que per validar offline fem
servir el més venut per regió en tot el període d'entrenament com a aproximació.

In [4]:
region_rows = sql(f"""
    SELECT region::String AS region, product_id::String AS product_id, count() AS n
    FROM shop.history_events WHERE event_type = 'purchase' AND ts < '{CUT}'
    GROUP BY region, product_id ORDER BY n DESC LIMIT 20 BY region""")
region_trending = {}
for r in region_rows:
    region_trending.setdefault(r["region"], []).append(r["product_id"])
print(len(region_trending), "regions amb tendència")

10 regions amb tendència


## Policies

Una policy és una funció `row -> [10 product_ids]`. `baseline` i `coview` són les de
l'starter; `combinada` hi afegeix historial d'usuari i tendència regional.

In [5]:
def baseline(row):
    return [p for p in best][:10]

def coview(row):
    score = Counter()
    for k, v in enumerate(reversed(row.recent_views)):
        for rank, q in enumerate(dict(covisit.get(v["product_id"], [])).keys()):
            score[q] += 0.7 ** k / (rank + 1)
    items = [q for q, _ in score.most_common(10)]
    return (items + [p for p in baseline(row) if p not in items])[:10]

def combinada(row):
    score = Counter()
    for k, v in enumerate(reversed(row.recent_views)):
        for rank, (q, _w) in enumerate(covisit.get(v["product_id"], []), start=1):
            score[q] += 2.0 * 0.7 ** k / rank
    if row.user_id in user_profiles:
        for p, n in user_profiles[row.user_id]:
            score[p] += 1.0 * n
    region = (row.context or {}).get("region")
    for rank, p in enumerate(region_trending.get(region, []), start=1):
        score[p] += 0.5 / rank
    items = [p for p, _ in score.most_common(10)]
    return (items + [p for p in baseline(row) if p not in items])[:10]

POLICIES = {"baseline": baseline, "coview": coview, "combinada": combinada}

## Evaluate (no es toca)

In [6]:
def evaluate(policy, df):
    recs = [policy(r) for r in df.itertuples()]
    clicked = df.clicked.to_numpy()
    has_click = df.clicked.notna().to_numpy()
    hit = np.array([c in r for c, r in zip(clicked, recs)])[has_click].mean()
    first_shown = df.shown.str[0].to_numpy()
    match = np.array([r[0] == s for r, s in zip(recs, first_shown)])
    reward = (clicked == first_shown).astype(float)
    w = match / df.propensity.str[0].to_numpy()
    return {"hit_rate@10": hit, "IPS_ctr_slot1": (w * reward).mean(),
            "SNIPS_ctr_slot1": (w * reward).sum() / max(w.sum(), 1e-9), "slot1_matches": int(match.sum())}

old_slot1 = (test.clicked == test.shown.str[0]).mean()
print(f"sistema antic, CTR observat slot-1: {old_slot1:.4f}")
pd.DataFrame({name: evaluate(p, test) for name, p in POLICIES.items()}).T

sistema antic, CTR observat slot-1: 0.0112


,hit_rate@10,IPS_ctr_slot1,SNIPS_ctr_slot1,slot1_matches
baseline,0.392924,0.006998,0.047035,3997.0
coview,0.274993,0.014135,0.019799,6022.0
combinada,0.405521,0.014579,0.054888,1879.0


In [ ]:
import sys
sys.path.insert(0, "..")
from src.recommender import Models, recommend as recommend_v3
from src.features import LiveState

# Carreguem els models entrenats NOMÉS amb dades anteriors a CUT
models_eval = Models(suffix=f"_{CUT}")
live_eval = LiveState()  # Estat buit per a l'avaluació històrica offline

def v3_hybrid(row):
    req = {
        "request_id": row.request_id,
        "user_id": row.user_id,
        "is_new_user": False,
        "context": row.context if isinstance(row.context, dict) else {},
        "recent_views": row.recent_views if isinstance(row.recent_views, list) else [],
    }
    return recommend_v3(req, models_eval, live_eval)

POLICIES = {"baseline": baseline, "coview": coview, "v3_hybrid": v3_hybrid}

## Com decidir

- Mireu sobretot **SNIPS_ctr_slot1** (és la mètrica menys esbiaixada i menys sorollosa
  que l'IPS cru). **hit_rate@10** confirma la tendència general però afavoreix els
  productes populars.
- Si **slot1_matches** és baix per alguna policy, no us fieu massa de l'IPS/SNIPS
  d'aquella fila: hi ha poques mostres.
- Si `combinada` guanya a `baseline` i `coview`, porteu els mateixos pesos
  (`W_SESSION_COVIEW`, `W_USER_PROFILE`, `W_REGION_TREND`) a `src/recommender.py`.
  Si no guanya, proveu d'ajustar els pesos aquí abans de descartar la idea.